# 01. Сборка датасета лиц из кадров роликов

Кадры → **YuNet** (тот же детектор и те же веса, что в `cv-service`) → выравнивание по глазам и кроп с запасом →
дедупликация почти одинаковых кадров → разбиение train/val/test **по роликам** → подсказки от предобученной
HSEmotion → ручная разметка эмоций → `datasets/faces/labels.csv`.

**Важно:** YuNet находит *лица*, а не *эмоции*. Рамки и кропы получаются автоматически, а метку эмоции
ставит человек. Подсказки модели только ускоряют работу, и для test их нужно скрывать: иначе
разметка подстроится под модель «до», и сравнение «до/после» станет нечестным.

### Где лежат данные

```
datasets/                      # в .gitignore
├── videos/                    # (необязательно) исходные ролики: clip_01.mp4, ...
├── raw/                       # кадры, по папке на ролик
│   ├── clip_01/000000.jpg
│   └── clip_02/...
├── weights/                   # кэш весов HSEmotion
└── faces/                     # результат этого ноутбука
    ├── crops/clip_01/000000_f0.jpg
    ├── labels.csv
    └── classes.json
```

Папка на ролик нужна, чтобы кадры одного ролика не попали одновременно в train и test.
Соседние кадры почти одинаковые, и такая утечка завысит метрики.

In [ ]:
# Зависимости (если запускаете не в .venv проекта или в Colab):
# %pip install opencv-python-headless numpy pandas matplotlib tqdm ipywidgets pillow
# Для подсказок от HSEmotion дополнительно: %pip install torch torchvision timm

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Корень проекта: ноутбук лежит в notebooks/. В Colab укажите путь на Google Drive через EMO_PROJECT_ROOT.
_default_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT = Path(os.environ.get("EMO_PROJECT_ROOT", _default_root)).resolve()

DATASETS = PROJECT_ROOT / "datasets"
VIDEOS_DIR = DATASETS / "videos"
RAW_DIR = DATASETS / "raw"
OUT_DIR = DATASETS / "faces"
CROPS_DIR = OUT_DIR / "crops"
LABELS_CSV = OUT_DIR / "labels.csv"
WEIGHTS_DIR = DATASETS / "weights"
YUNET_PATH = PROJECT_ROOT / "cv-service/src/model/weights/face_detection_yunet_2023mar.onnx"

# Набор классов. Порядок фиксируется в classes.json и дальше используется при обучении.
# Если contempt почти не встречается, уберите его (и возьмите веса enet_b2_7 или оставьте 8-классовые).
CLASSES = ["anger", "contempt", "disgust", "fear", "happiness", "neutral", "sadness", "surprise"]
SPECIAL_LABELS = ["unsure", "bad_crop"]  # в обучение не идут

# Детекция
SCORE_THR = 0.7        # в cv-service 0.6; для датасета берём только уверенные лица
MIN_FACE_PX = 48       # меньшие лица малоинформативны для эмоций
MARGIN = 0.2           # запас вокруг рамки (лоб, подбородок)
CROP_SIZE = 260        # вход EfficientNet-B2; для B0 изображение уменьшат при загрузке

# Видео → кадры (если используете videos/)
EXTRACT_FPS = 2.0

# Дедупликация: расстояние Хэмминга между 64-битными dHash кропов одного ролика
DHASH_MAX_DIST = 3     # 0 = выключить

# Разбиение по роликам (детерминированно по имени ролика: новые ролики не перетасовывают старые)
SPLIT_SEED = "v1"
TEST_PCT, VAL_PCT = 20, 15
SPLIT_OVERRIDES: dict[str, str] = {}   # {"clip_07": "test"} — ручная правка

assert YUNET_PATH.exists(), YUNET_PATH
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 1. (необязательно) Нарезка кадров из видео

In [ ]:
def extract_frames(video: Path, out_dir: Path, fps: float) -> int:
    cap = cv.VideoCapture(str(video))
    src_fps = cap.get(cv.CAP_PROP_FPS) or 25.0
    step = max(int(round(src_fps / fps)), 1)
    out_dir.mkdir(parents=True, exist_ok=True)
    i = saved = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % step == 0:
            cv.imwrite(str(out_dir / f"{i:06d}.jpg"), frame, [cv.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        i += 1
    cap.release()
    return saved


videos = sorted(p for p in VIDEOS_DIR.glob("*") if p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv", ".webm"})
for v in videos:
    target = RAW_DIR / v.stem
    if target.exists():
        print(f"skip {v.name}: {target} уже есть")
        continue
    print(f"{v.name}: {extract_frames(v, target, EXTRACT_FPS)} кадров")
print(f"видео: {len(videos)}")

## 2. Детекция, выравнивание и кропы

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
frames = sorted(p for p in RAW_DIR.rglob("*") if p.suffix.lower() in IMG_EXT)


def clip_of(frame: Path) -> str:
    rel = frame.relative_to(RAW_DIR)
    if len(rel.parts) == 1:
        # Кадр лежит прямо в raw/: считаем его отдельным роликом (утечку между кадрами одного ролика не поймаем).
        return rel.stem
    return "/".join(rel.parts[:-1])


loose = sum(1 for f in frames if len(f.relative_to(RAW_DIR).parts) == 1)
print(f"кадров: {len(frames)}, роликов: {len({clip_of(f) for f in frames})}")
if loose:
    print(f"ВНИМАНИЕ: {loose} кадров лежат прямо в raw/ без папки ролика — разложите их по папкам.")

In [ ]:
detector = cv.FaceDetectorYN.create(str(YUNET_PATH), "", (320, 320), SCORE_THR, 0.3, 5000)


def detect(img: np.ndarray) -> np.ndarray:
    # Строка на лицо: x, y, w, h, правый глаз, левый глаз, нос, правый и левый угол рта (x, y), score.
    h, w = img.shape[:2]
    detector.setInputSize((w, h))
    _, faces = detector.detect(img)
    return np.empty((0, 15), np.float32) if faces is None else faces


def align_crop(img: np.ndarray, face: np.ndarray, margin: float, size: int) -> np.ndarray:
    # Поворачиваем так, чтобы глаза были на одной горизонтали, и вырезаем квадрат с запасом
    # одним аффинным преобразованием.
    x, y, w, h = face[:4]
    (rx, ry), (lx, ly) = face[4:6], face[6:8]
    angle = float(np.degrees(np.arctan2(ly - ry, lx - rx)))
    cx, cy = x + w / 2, y + h / 2
    side = max(w, h) * (1 + 2 * margin)
    M = cv.getRotationMatrix2D((float(cx), float(cy)), angle, size / side)
    M[0, 2] += size / 2 - cx
    M[1, 2] += size / 2 - cy
    return cv.warpAffine(img, M, (size, size), flags=cv.INTER_LINEAR, borderMode=cv.BORDER_REPLICATE)


def dhash(img: np.ndarray, n: int = 8) -> int:
    g = cv.resize(cv.cvtColor(img, cv.COLOR_BGR2GRAY), (n + 1, n), interpolation=cv.INTER_AREA)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


rows = []
for frame in tqdm(frames, desc="YuNet"):
    img = cv.imread(str(frame))
    if img is None:
        print("не читается:", frame)
        continue
    clip = clip_of(frame)
    for i, f in enumerate(detect(img)):
        x, y, w, h = f[:4]
        if min(w, h) < MIN_FACE_PX:
            continue
        crop = align_crop(img, f, MARGIN, CROP_SIZE)
        rel = Path("crops") / clip / f"{frame.stem}_f{i}.jpg"
        (OUT_DIR / rel).parent.mkdir(parents=True, exist_ok=True)
        cv.imwrite(str(OUT_DIR / rel), crop, [cv.IMWRITE_JPEG_QUALITY, 95])
        rows.append({
            "crop_path": rel.as_posix(), "clip_id": clip,
            "frame_path": frame.relative_to(DATASETS).as_posix(), "face_idx": i,
            "x": int(x), "y": int(y), "w": int(w), "h": int(h),
            "det_score": round(float(f[14]), 4), "dhash": dhash(crop),
        })

faces = pd.DataFrame(rows)
print(f"лиц: {len(faces)} на {faces['frame_path'].nunique() if len(faces) else 0} кадрах из {len(frames)}")

### Дедупликация

Соседние кадры ролика дают почти одинаковые кропы. Они раздувают датасет, не добавляя информации,
и тратят время на разметку. Дубли не удаляются, а помечаются `is_dup=True` и по умолчанию не попадают в очередь разметки.

In [ ]:
def hamming(a: int, b: int) -> int:
    return (a ^ b).bit_count()


faces["is_dup"] = False
if DHASH_MAX_DIST > 0 and len(faces):
    for clip, grp in faces.sort_values("crop_path").groupby("clip_id"):
        kept: list[int] = []
        for idx, h in grp["dhash"].items():
            if any(hamming(h, k) <= DHASH_MAX_DIST for k in kept):
                faces.at[idx, "is_dup"] = True
            else:
                kept.append(h)
print(f"дублей: {faces['is_dup'].sum()} из {len(faces)}")

## 3. Контроль качества детекции

In [ ]:
BLUE = "#2a78d6"


def show_frames_with_boxes(n: int = 6, seed: int = 0) -> None:
    if faces.empty:
        print("лиц нет")
        return
    sample = faces["frame_path"].drop_duplicates().sample(min(n, faces["frame_path"].nunique()), random_state=seed)
    cols = 3
    fig, axes = plt.subplots((len(sample) + cols - 1) // cols, cols, figsize=(5 * cols, 3.6 * ((len(sample) + cols - 1) // cols)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, fp in zip(axes.flat, sample):
        img = cv.cvtColor(cv.imread(str(DATASETS / fp)), cv.COLOR_BGR2RGB)
        ax.imshow(img)
        for _, r in faces[faces["frame_path"] == fp].iterrows():
            ax.add_patch(plt.Rectangle((r.x, r.y), r.w, r.h, fill=False, lw=2, ec=BLUE))
            ax.text(r.x, r.y - 4, f"{r.det_score:.2f}", color="white", fontsize=8,
                    bbox=dict(fc=BLUE, ec="none", pad=1))
        ax.set_title(fp, fontsize=8)
    plt.tight_layout()
    plt.show()


show_frames_with_boxes()

In [ ]:
if len(faces):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
    axes[0].hist(faces["det_score"], bins=30, color=BLUE, edgecolor="white")
    axes[0].set_title("Уверенность детектора")
    axes[1].hist(np.minimum(faces["w"], faces["h"]), bins=30, color=BLUE, edgecolor="white")
    axes[1].set_title("Размер лица, px (min(w, h))")
    per_clip = faces[~faces["is_dup"]].groupby("clip_id").size().sort_values()
    axes[2].barh(per_clip.index[-20:], per_clip.values[-20:], color=BLUE)
    axes[2].set_title("Уникальных лиц на ролик (топ-20)")
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


def show_crops(df: pd.DataFrame, n: int = 24, title_col: str | None = None, seed: int = 0) -> None:
    if df.empty:
        print("пусто")
        return
    s = df.sample(min(n, len(df)), random_state=seed)
    cols = 8
    rows_n = (len(s) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(2 * cols, 2.2 * rows_n), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, s.iterrows()):
        ax.imshow(cv.cvtColor(cv.imread(str(OUT_DIR / r.crop_path)), cv.COLOR_BGR2RGB))
        ax.set_title(str(r[title_col]) if title_col else r.clip_id, fontsize=8)
    plt.tight_layout()
    plt.show()


show_crops(faces[~faces["is_dup"]] if len(faces) else faces)

## 4. Подсказки от предобученной HSEmotion (необязательно)

Предсказания (`pred_label`, `pred_conf`) нужны только для удобства разметки train/val: близкие
эмоции идут подряд, а в спорных случаях есть подсказка. В test подсказки скрыты.
Нужны `torch`, `torchvision` и `timm`. Ячейку можно пропустить.

In [ ]:
USE_PSEUDO_LABELS = True
PSEUDO_MODEL = "enet_b0_8_best_vgaf"   # B0 быстрее на CPU

In [ ]:
import torch
# --- Загрузка предобученной HSEmotion (EfficientNet, AffectNet) ---------------------------
# Порядок классов HSEmotion (idx_to_class из пакета hsemotion).
HSE_CLASSES_8 = ["anger", "contempt", "disgust", "fear", "happiness", "neutral", "sadness", "surprise"]
HSE_CLASSES_7 = ["anger", "disgust", "fear", "happiness", "neutral", "sadness", "surprise"]
HSE_MODELS = {
    # имя весов           : (архитектура timm, размер входа, классы)
    "enet_b0_8_best_vgaf": ("tf_efficientnet_b0", 224, HSE_CLASSES_8),
    "enet_b0_8_best_afew": ("tf_efficientnet_b0", 224, HSE_CLASSES_8),
    "enet_b0_8_va_mtl":    ("tf_efficientnet_b0", 224, HSE_CLASSES_8),  # +2 выхода valence/arousal
    "enet_b2_8":           ("tf_efficientnet_b2", 260, HSE_CLASSES_8),
    "enet_b2_7":           ("tf_efficientnet_b2", 260, HSE_CLASSES_7),
}
HSE_URL = "https://github.com/av-savchenko/face-emotion-recognition/raw/main/models/affectnet_emotions/{name}.pt"
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def load_hsemotion(name: str, weights_dir: Path) -> tuple["torch.nn.Module", int, list[str]]:
    """Возвращает (модель timm с весами HSEmotion, размер входа, классы в порядке выходов).

    Автор выложил веса как целиком сохранённую модель (pickle), поэтому первая загрузка
    идёт с weights_only=False — это исполняет код из файла, качайте только из репозитория автора.
    Дальше веса кэшируются как обычный state_dict и грузятся безопасно (weights_only=True).
    """
    import timm
    arch, size, classes = HSE_MODELS[name]
    weights_dir.mkdir(parents=True, exist_ok=True)
    sd_path = weights_dir / f"{name}.state_dict.pth"
    if not sd_path.exists():
        pt_path = weights_dir / f"{name}.pt"
        if not pt_path.exists():
            print("downloading", HSE_URL.format(name=name))
            torch.hub.download_url_to_file(HSE_URL.format(name=name), str(pt_path))
        legacy = torch.load(pt_path, map_location="cpu", weights_only=False)
        sd = {k.replace("classifier.0.", "classifier."): v for k, v in legacy.state_dict().items()}
        torch.save(sd, sd_path)
    sd = torch.load(sd_path, map_location="cpu", weights_only=True)
    model = timm.create_model(arch, pretrained=False, num_classes=sd["classifier.weight"].shape[0])
    model.load_state_dict(sd, strict=True)
    return model.eval(), size, classes


def restrict_head(model: "torch.nn.Module", src_classes: list[str], dst_classes: list[str]) -> "torch.nn.Module":
    """Оставляет в последнем слое только строки нужных классов (в порядке dst_classes).

    Так дообучение стартует ровно с предобученной модели, а у VA-моделей отбрасываются
    выходы valence/arousal.
    """
    missing = [c for c in dst_classes if c not in src_classes]
    assert not missing, f"классов {missing} нет в предобученной модели: {src_classes}"
    idx = torch.tensor([src_classes.index(c) for c in dst_classes])
    old = model.classifier
    new = torch.nn.Linear(old.in_features, len(dst_classes))
    with torch.no_grad():
        new.weight.copy_(old.weight[idx])
        new.bias.copy_(old.bias[idx])
    model.classifier = new
    return model

In [ ]:
if USE_PSEUDO_LABELS and len(faces):
    from PIL import Image
    from torchvision.transforms import v2 as T

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    hse, size, hse_classes = load_hsemotion(PSEUDO_MODEL, WEIGHTS_DIR)
    hse = restrict_head(hse, hse_classes, CLASSES).to(device)
    tf = T.Compose([T.Resize((size, size)), T.ToImage(), T.ToDtype(torch.float32, scale=True),
                    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    preds, confs = [], []
    paths = faces["crop_path"].tolist()
    with torch.inference_mode():
        for i in tqdm(range(0, len(paths), 64), desc="HSEmotion"):
            x = torch.stack([tf(Image.open(OUT_DIR / p).convert("RGB")) for p in paths[i:i + 64]]).to(device)
            prob = hse(x).softmax(1).cpu()
            c, k = prob.max(1)
            preds += [CLASSES[j] for j in k.tolist()]
            confs += c.tolist()
    faces["pred_label"], faces["pred_conf"] = preds, np.round(confs, 4)
    print(faces["pred_label"].value_counts())
else:
    faces["pred_label"], faces["pred_conf"] = None, np.nan

## 5. Разбиение по роликам и сохранение `labels.csv`

Сплит назначается по хэшу имени ролика, поэтому стабилен между перезапусками. При повторном запуске
уже поставленные метки (`label`) и сплиты сохраняются: ноутбук можно прогонять заново после добавления новых роликов.

In [ ]:
def split_of(clip: str) -> str:
    if clip in SPLIT_OVERRIDES:
        return SPLIT_OVERRIDES[clip]
    bucket = int(hashlib.md5(f"{SPLIT_SEED}:{clip}".encode()).hexdigest(), 16) % 100
    return "test" if bucket < TEST_PCT else "val" if bucket < TEST_PCT + VAL_PCT else "train"


faces["split"] = faces["clip_id"].map(split_of) if len(faces) else []
faces["label"] = None

if LABELS_CSV.exists():
    old = pd.read_csv(LABELS_CSV, dtype={"label": "string", "split": "string"})
    keep = old.set_index("crop_path")
    known = faces["crop_path"].isin(keep.index)
    faces.loc[known, "label"] = faces.loc[known, "crop_path"].map(keep["label"])
    labeled_split = keep.loc[keep["label"].notna(), "split"]
    has_old_split = faces["crop_path"].isin(labeled_split.index)
    faces.loc[has_old_split, "split"] = faces.loc[has_old_split, "crop_path"].map(labeled_split)
    lost = set(old.loc[old["label"].notna(), "crop_path"]) - set(faces["crop_path"])
    print(f"перенесено меток: {faces['label'].notna().sum()}; потеряно (кропа больше нет): {len(lost)}")

faces = faces.drop(columns=["dhash"]).sort_values(["split", "clip_id", "crop_path"]).reset_index(drop=True)
faces.to_csv(LABELS_CSV, index=False)
(OUT_DIR / "classes.json").write_text(json.dumps(CLASSES, ensure_ascii=False, indent=2))

summary = faces[~faces["is_dup"]].groupby("split").agg(clips=("clip_id", "nunique"), faces=("crop_path", "size"))
print(summary)
print("сохранено:", LABELS_CSV)

## 6. Разметка

Мини-разметчик на `ipywidgets`: кнопка = метка, CSV сохраняется после каждого клика.
`unsure` — не уверены, `bad_crop` — не лицо, размыто или перекрыто (в обучение и тест не попадут).

Порядок работы:
1. **Сначала test** (`LABEL_SPLIT = "test"`), подсказки скрыты. Это эталон, на котором сравниваются модели «до» и «после».
   Лучше, если его размечают двое и спорные кадры отбрасываются.
2. Затем val и train, подсказки можно включить.

Если удобнее внешний инструмент (Label Studio, CVAT), выгрузите `crop_path` и верните CSV с колонкой `label`.

In [ ]:
LABEL_SPLIT = "test"          # "test" | "val" | "train"
INCLUDE_DUPS = False
SHOW_HINTS = LABEL_SPLIT != "test"

In [ ]:
import ipywidgets as W
from IPython.display import display

df = pd.read_csv(LABELS_CSV, dtype={"label": "string"})
mask = (df["split"] == LABEL_SPLIT) & df["label"].isna()
if not INCLUDE_DUPS:
    mask &= ~df["is_dup"]
order = df[mask].sort_values(["pred_label", "pred_conf"] if SHOW_HINTS else ["clip_id", "crop_path"])
queue = order.index.tolist()
state = {"pos": 0, "history": []}

img_w = W.Image(format="jpeg", width=300, height=300)
info = W.HTML()
buttons = [W.Button(description=c, layout=W.Layout(width="110px")) for c in CLASSES]
extra = [W.Button(description=c, button_style="warning", layout=W.Layout(width="110px")) for c in SPECIAL_LABELS]
back = W.Button(description="← назад", layout=W.Layout(width="110px"))


def render() -> None:
    done = int(df.loc[df["split"] == LABEL_SPLIT, "label"].notna().sum())
    total = int(((df["split"] == LABEL_SPLIT) & (INCLUDE_DUPS | ~df["is_dup"])).sum())
    if state["pos"] >= len(queue):
        info.value = f"<b>Готово</b>: размечено {done}/{total} в {LABEL_SPLIT}"
        img_w.value = b""
        return
    r = df.loc[queue[state["pos"]]]
    img_w.value = (OUT_DIR / r.crop_path).read_bytes()
    hint = f" · подсказка: <b>{r.pred_label}</b> ({r.pred_conf:.2f})" if SHOW_HINTS and pd.notna(r.pred_label) else ""
    info.value = f"{LABEL_SPLIT}: {done}/{total} · {r.crop_path}{hint}"


def on_label(label: str):
    def handler(_):
        if state["pos"] >= len(queue):
            return
        idx = queue[state["pos"]]
        df.at[idx, "label"] = label
        df.to_csv(LABELS_CSV, index=False)
        state["history"].append(idx)
        state["pos"] += 1
        render()
    return handler


def on_back(_):
    if state["history"]:
        idx = state["history"].pop()
        df.at[idx, "label"] = pd.NA
        df.to_csv(LABELS_CSV, index=False)
        state["pos"] -= 1
        render()


for b in buttons + extra:
    b.on_click(on_label(b.description))
back.on_click(on_back)
render()
display(W.VBox([info, img_w, W.HBox(buttons[:4]), W.HBox(buttons[4:]), W.HBox(extra + [back])]))

## 7. Итоги разметки

In [ ]:
df = pd.read_csv(LABELS_CSV, dtype={"label": "string"})
labeled = df[df["label"].notna()]
if labeled.empty:
    print("пока ничего не размечено")
print(f"размечено: {len(labeled)} из {len(df)} (без дублей: {(~df['is_dup']).sum()})")
table = pd.crosstab(labeled["label"], labeled["split"]).reindex(CLASSES + SPECIAL_LABELS, fill_value=0)
display(table)

# Согласие подсказок с ручной разметкой в train/val — грубая оценка модели «до».
# Честная оценка будет в 02_train_eval на test.
tv = labeled[labeled["split"].isin(["train", "val"]) & labeled["label"].isin(CLASSES) & labeled["pred_label"].notna()]
if len(tv):
    print(f"совпадение подсказок с разметкой (train+val): {(tv['label'] == tv['pred_label']).mean():.1%}")

low = table.loc[CLASSES].min(axis=1)
print("классы, где меньше 20 примеров в каком-то сплите:", low[low < 20].index.tolist())